# 09 · From the perceptron to a tuned classifier

The practical part of lab 09. On data from the inspection of moulded parts, you train a perceptron and a logistic neuron,
evaluate them with the metrics from the lecture, **tune them on your own**, choose the decision threshold by the cost of errors
and test the result once. You have seen every setting you tune here as a plot in the educational part `perceptron-visual-en.ipynb`.

Short runnable examples use small illustrative data and variables prefixed with `demo_`. Run them in order:
they build on each other and show the syntax of the whole workflow. Write your solution in the empty code cells below the tasks.
The test set is opened only in step 7. When you are done, run `Restart Kernel and Run All` to check that your solution runs in order.

## Environment setup and data loading

1. Open the notebook in Jupyter (the README describes the packages) or in Google Colab.
2. Run the example below. It imports every tool you will need and prepares small illustrative `demo_` data.
3. The prepared cell below it loads the data into the table `raw` and sets `SEED = 42` for all random steps. It uses the local file
   `data/parts.csv` when the notebook runs in the lab folder or the repository root; otherwise it downloads the same file from GitHub
   (internet access needed).

### Example: imports and illustrative data

In [ ]:
import warnings                                              # Silence warnings about unfinished learning (see below).
from IPython.display import display                          # Display tables in your own tasks.
import matplotlib.pyplot as plt                              # Plots.
import numpy as np                                           # Numerical operations.
import pandas as pd                                          # Data and result tables.
from sklearn.base import clone                               # A fresh unfitted copy of a model configuration.
from sklearn.model_selection import train_test_split         # Split inputs and targets together.
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, cross_val_predict  # Grid search and cross-validation.
from sklearn.pipeline import Pipeline                        # Chain several steps into one model.
from sklearn.preprocessing import StandardScaler, PolynomialFeatures  # Standardisation and new product features.
from sklearn.linear_model import Perceptron, SGDClassifier  # Perceptron and a neuron learned by stochastic gradient descent.
from sklearn.dummy import DummyClassifier                    # A simple reference model.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # Confusion matrix as numbers and as a plot.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score  # Metrics of hard decisions.
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay  # Metrics and curves over all thresholds.
from sklearn.exceptions import ConvergenceWarning            # Warning type for learning stopped by the epoch limit.

warnings.filterwarnings("ignore", category=ConvergenceWarning)  # Small rates or few epochs are tuned on purpose; we judge the result, not the warning.

demo_rng = np.random.default_rng(7)                          # Illustrative data, not the assignment data.
demo_X = pd.DataFrame({"size": demo_rng.normal(50, 10, 300),       # A feature in tens.
                       "weight": demo_rng.normal(2000, 300, 300)})  # A feature in thousands.
demo_risk = -2.2 + 0.09 * (demo_X["size"] - 50) + 0.004 * (demo_X["weight"] - 2000)  # Hidden risk of class 1.
demo_y = (demo_rng.random(300) < 1 / (1 + np.exp(-demo_risk))).astype(int)  # About 15 % of class 1.
demo_X_dev, demo_X_test, demo_y_dev, demo_y_test = train_test_split(demo_X, demo_y, test_size=0.25, stratify=demo_y, random_state=7)  # A quarter for the test.
demo_X_train, demo_X_val, demo_y_train, demo_y_val = train_test_split(demo_X_dev, demo_y_dev, test_size=0.25, stratify=demo_y_dev, random_state=7)  # A quarter of the rest for validation.
print(len(demo_X_train), len(demo_X_val), len(demo_X_test), "class 1 share:", round(demo_y.mean(), 3))  # Sizes of the three parts.

**Prepared code:**

In [ ]:
from pathlib import Path                                     # Paths to a local copy of the data.
SEED = 42                                                    # The same random split and learning for everyone.
DATA_URL = ("https://raw.githubusercontent.com/CVUT-FS-12110/Artificial-Intelligence-and-Neural-Networks/"
            "master/09-Perceptron/data/parts.csv")           # The same file in the course repository on GitHub.
local = [Path("data/parts.csv"), Path("09-Perceptron/data/parts.csv")]  # The lab folder or the repository root.
found = [path for path in local if path.exists()]           # Local copies that exist.
source = found[0] if found else DATA_URL                     # Without a local copy (e.g. Google Colab) download from GitHub.
raw = pd.read_csv(source)                                    # 2000 inspected parts.
print("data loaded from:", source)          # Local file or URL.
display(raw.head())  # The first five rows.

## Problem and data

Each row is one **injection-moulded part** with eight process settings measured during its production and the result of the
final inspection `defect` (1 = defective). The data are synthetic; their description is in `data/README.md`.

| Column | Meaning | Typical value |
|---|---|---|
| `melt_temp` | melt temperature | 230 °C |
| `pressure` | injection pressure | 900 bar |
| `cooling_time` | cooling time | 12 s |
| `mould_temp` | mould temperature | 60 °C |
| `humidity` | humidity in the granulate store | 45 % |
| `vibration` | press vibration | 0.5 mm/s |
| `shot_weight` | shot weight | 52 g |
| `cycle_time` | cycle time | 30 s |
| `shrinkage` | measured shrinkage – not used in this lab, **not an input** | 1.5 % |

**Goal:** predict the inspection result from the settings, so that suspicious parts are sent to a detailed check.
**Costs:** a defective part shipped to a customer (FN) costs **10** units (complaint), a good part sent to the detailed check
(FP) costs **1** unit. The quality of a decision is therefore measured by the **total cost** C = 10·FN + 1·FP,
not by accuracy alone.

## 1. Inspection (prepared)

**Output:** the share of defective parts and a table of feature ranges. Notice how rare class 1 is and how different the scales are.

**Prepared code:**

In [ ]:
print("rows:", len(raw), "| missing values:", int(raw.isna().sum().sum()))  # Size and completeness.
print("share of defective parts:", round(raw["defect"].mean(), 3))  # Share of class 1.
display(raw.drop(columns=["part_id", "defect"]).describe().T[["mean", "std", "min", "max"]].round(3))  # Ranges of the inputs.

## 2. Features and a fixed split (prepared)

**Output:** `X` (eight inputs) and `y` (`defect`); a development part `X_dev`, `y_dev` (1500 parts) and a test set `X_test`, `y_test`
(500 parts) that stays closed until step 7. The development part is further split into training `X_train`, `y_train` (1125)
and validation `X_val`, `y_val` (375). All splits are stratified: every part keeps the share of defects.

**Prepared code:**

In [ ]:
FEATURES = ["melt_temp", "pressure", "cooling_time", "mould_temp", "humidity", "vibration", "shot_weight", "cycle_time"]  # The eight inputs.
X = raw[FEATURES]                                            # Eight inputs; part_id and shrinkage are not inputs.
y = raw["defect"]                                            # Target class.
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)  # A quarter for the test, stratified.
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)  # A quarter of the rest for validation.
for name, part in (("train", y_train), ("validation", y_val), ("dev", y_dev), ("test", y_test)):  # Parts and defects in each part.
    print(f"{name:10s} {len(part):5d} parts, {int(part.sum()):4d} defective")  # One line per part.

## 3. Reference model and evaluation function (prepared)

**Output:** a function `evaluate(y_true, y_pred)` that returns the confusion-matrix counts, five metrics and the total cost,
and a table `reference` for the model that always predicts the most frequent class (“every part is good”).
Use `evaluate` in all following steps, so the results can be compared.

**Prepared code:**

In [ ]:
C_FN, C_FP = 10, 1                                           # Cost of a missed defect and of a false alarm.


def evaluate(y_true, y_pred):  # Used in all following steps.
    # Counts of the confusion matrix, metrics and total cost of hard decisions 0/1.
    y_pred = np.asarray(y_pred).astype(int)                  # Also accepts True/False.
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()  # Four counts in this order.
    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),  # Counts of the confusion matrix.
            "accuracy": accuracy_score(y_true, y_pred),  # (TP + TN) / n
            "precision": precision_score(y_true, y_pred, zero_division=0),  # 0 when there is no alarm
            "recall": recall_score(y_true, y_pred),  # TP / (TP + FN)
            "F1": f1_score(y_true, y_pred, zero_division=0),  # harmonic mean of precision and recall
            "balanced accuracy": balanced_accuracy_score(y_true, y_pred),  # (recall + specificity) / 2
            "cost": int(C_FN * fn + C_FP * fp)}  # C = 10·FN + 1·FP


baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)  # Always predicts “good”.
reference = pd.DataFrame([evaluate(y_val, baseline.predict(X_val))], index=["always good"])  # Validation result of the reference.
display(reference.round(3))  # Rounded table.

**Think about:** The reference model has an accuracy of about 0.89. Why is it still useless? Which metrics and which cost show it?

## 4. Perceptron

**Goal / output:** a fitted `perceptron_pipe`, its validation predictions `val_pred_perceptron`, a confusion matrix, a table
`perceptron_results` comparing the scaled and the unscaled perceptron, and a list `epoch_errors` with the training errors after each of 30 epochs.

1. `perceptron_pipe` = `Pipeline` with the steps `scale`: `StandardScaler()` → `model`: `Perceptron(random_state=SEED)`.
   Fit it on the training data and predict the validation data.
2. Draw the confusion matrix of the validation predictions. From the four counts compute **by hand** precision and recall
   and check them against `evaluate`.
3. Fit the same perceptron **without** scaling (only `Perceptron(random_state=SEED)`). Put both results of `evaluate`
   into `perceptron_results` together with `reference`.
4. Training errors per epoch: fit a `StandardScaler` on `X_train`, create a new `Perceptron(random_state=SEED)` and call
   `partial_fit` 30 times on the scaled training data (one call = one epoch). After each epoch store the number of training errors
   in `epoch_errors` and plot the list.

**Use:** `Pipeline`, `StandardScaler`, `Perceptron`, `.fit`, `.predict`, `ConfusionMatrixDisplay.from_predictions`,
`confusion_matrix(...).ravel()`, `evaluate`, `pd.DataFrame`, `.partial_fit(X, y, classes=[0, 1])`, `plt.plot`.

<details><summary>Hint</summary>

`tn, fp, fn, tp = confusion_matrix(y_val, val_pred).ravel()` gives the four counts in this order. `Perceptron` learns with the
rule from the lecture (an update after each misclassified sample, in a random order in each epoch) and stops when the training error
no longer decreases. `partial_fit` makes exactly one pass over the data and continues from the current weights; the first call needs
the list of all classes. The scaler is fitted on the training data only and then transforms them:
`scaler.transform(X_train)`.

</details>

**Think about:** Does the number of training errors settle as the epochs go on? What does that say about the data
(educational part, section 3)? Which metric differs most between the scaled and the unscaled perceptron?

### Example: perceptron, confusion matrix and errors per epoch

In [ ]:
demo_pipe = Pipeline([("scale", StandardScaler()), ("model", Perceptron(random_state=7))])  # Scaling, then the perceptron.
demo_pipe.fit(demo_X_train, demo_y_train)                    # The scaler and the weights are learned from training data only.
demo_pred = demo_pipe.predict(demo_X_val)  # Validation predictions 0/1.
ConfusionMatrixDisplay.from_predictions(demo_y_val, demo_pred, colorbar=False)  # Rows: true class, columns: prediction.
plt.show()  # Display the plot.
demo_tn, demo_fp, demo_fn, demo_tp = confusion_matrix(demo_y_val, demo_pred).ravel()  # The four counts.
print("recall by hand:", demo_tp / (demo_tp + demo_fn), "| evaluate:", evaluate(demo_y_val, demo_pred)["recall"])  # The same value both ways.

demo_scaler = StandardScaler().fit(demo_X_train)             # A separately fitted scaler for the epoch loop.
demo_Z = demo_scaler.transform(demo_X_train)  # Scaled training inputs.
demo_model, demo_errors = Perceptron(random_state=7), []  # A new perceptron and an empty list.
for demo_epoch in range(10):                                 # Ten epochs; one partial_fit call = one pass over the data.
    demo_model.partial_fit(demo_Z, demo_y_train, classes=[0, 1])  # One epoch.
    demo_errors.append(int((demo_model.predict(demo_Z) != demo_y_train).sum()))  # Training errors after the epoch.
print("training errors after epochs:", demo_errors)  # They need not decrease.

**Your code:**

## 5. Logistic neuron

**Goal / output:** a fitted `logistic_pipe`, the validation probabilities `val_proba` of class 1, a table `step5_results`
with the perceptron and the logistic neuron at the threshold 0.5, the numbers `val_auc` and `val_ap`, and three plots:
the ROC curves and the precision–recall curves of both models and a histogram of `val_proba` for each class.

1. `logistic_pipe` = `Pipeline`: `scale`: `StandardScaler()` → `model`:
   `SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.01, alpha=1e-4, random_state=SEED)`.
   It is one logistic neuron learned by stochastic gradient descent with a constant learning rate η = `eta0`
   and a small weight regularisation `alpha`.
2. Fit it on the training data, store `val_proba = logistic_pipe.predict_proba(X_val)[:, 1]` and the hard predictions at 0.5 (`.predict`).
   Add the result of `evaluate` to `step5_results` together with the scaled perceptron from step 4.
3. Compute `val_auc` (ROC AUC) and `val_ap` (area under the precision–recall curve) from `val_proba`. For the perceptron use its score
   `perceptron_pipe.decision_function(X_val)`: ROC and PR need only an ordering, not a probability.
4. Draw both ROC curves into one plot and both precision–recall curves into another. Draw a histogram of `val_proba` separately for
   good and defective parts.

**Use:** `SGDClassifier`, `.predict_proba`, `.decision_function`, `roc_auc_score`, `average_precision_score`,
`RocCurveDisplay.from_predictions`, `PrecisionRecallDisplay.from_predictions`, `plt.hist`.

<details><summary>Hint</summary>

Without `learning_rate="constant"` the classifier uses its own schedule and **ignores `eta0`** – the tuning in step 6 would then
change nothing. `predict_proba` returns two columns (class 0 and class 1); `[:, 1]` selects class 1. Both display functions take
`ax=` to draw into a prepared axis: `fig, axs = plt.subplots(1, 2)`, then `RocCurveDisplay.from_predictions(y_val, scores, name="...", ax=axs[0])`.
A histogram for one class: `plt.hist(val_proba[y_val == 1], bins=20)`.

</details>

**Think about:** The AUC is clearly above 0.5, yet at the threshold 0.5 the recall is low. Where in the histogram are most defective parts?
What would happen to recall, precision and cost if the threshold were 0.1?

### Example: probabilities, AUC and the curves

In [ ]:
demo_logistic = Pipeline([("scale", StandardScaler()),  # Scaling first …
                          ("model", SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.05, random_state=7))])  # One logistic neuron.
demo_logistic.fit(demo_X_train, demo_y_train)  # Learning on training data.
demo_proba = demo_logistic.predict_proba(demo_X_val)[:, 1]  # Probability of class 1.
print("AUC:", round(roc_auc_score(demo_y_val, demo_proba), 3), "| AP:", round(average_precision_score(demo_y_val, demo_proba), 3))  # Both measures use the whole ordering.
fig, axs = plt.subplots(1, 3, figsize=(14, 4))  # Three plots side by side.
RocCurveDisplay.from_predictions(demo_y_val, demo_proba, name="logistic", ax=axs[0], plot_chance_level=True)  # ROC curve with the diagonal of a random model.
PrecisionRecallDisplay.from_predictions(demo_y_val, demo_proba, name="logistic", ax=axs[1])  # Precision–recall curve.
axs[2].hist(demo_proba[demo_y_val == 0], bins=10, alpha=0.6, label="y = 0")  # Probabilities of class 0 …
axs[2].hist(demo_proba[demo_y_val == 1], bins=10, alpha=0.6, label="y = 1")  # … and of class 1.
axs[2].set(xlabel="probability of class 1")  # Axis label.
axs[2].legend()  # Legend.
plt.tight_layout()  # No overlapping labels.
plt.show()  # Display the plot.

**Your code:**

## 6. Independent tuning

In step 5 you used the settings given in the assignment. Now **tune them yourself**. Preprocessing and the neuron are tuned together.
There are three parts:

- **6.1 Manual tuning** on the fixed validation set: a few variants, each changing one thing. Quick and illustrative, but
  375 validation parts contain only 40 defects, so the differences are noisy.
- **6.2 Grid search** with five-fold cross-validation on the development data: systematic and more reliable.
- **6.3 Decision threshold** chosen by the cost of errors.

The tuning metric in 6.1 and 6.2 is **ROC AUC**. It does not depend on the threshold, so it judges how well the model orders the parts;
the threshold is chosen separately in 6.3. The **test set is not used** for tuning.

| Parameter | Possible values | Link to the educational part |
|---|---|---|
| `scale` | `"passthrough"`, `StandardScaler()` | 4 input scaling, 5 learning rate |
| `features` | `"passthrough"`, `PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)` | 10 XOR and the product x₁·x₂ |
| `model__loss` | `"perceptron"` (perceptron rule), `"log_loss"` (logistic neuron) | 1 and 6 |
| `model__eta0` | 0.0001, 0.001, 0.01, 0.1, 1 | 5 learning rate |
| `model__alpha` | 0.00001, 0.001, 0.01, 0.1 | 6 weight regularisation |

`"passthrough"` means “skip this step”. `PolynomialFeatures(..., interaction_only=True)` adds the products of all pairs of inputs
(28 new features). A double underscore separates the name of a step and its parameter: `model__eta0` is the parameter `eta0` of the step `model`.

### 6.1 Manual tuning on the fixed validation set

**Goal / output:** `tune_base` – an unfitted pipeline with the steps `scale`, `features` and `model`; a table `manual_results`
with at least six variants and their validation AUC; a dictionary `manual_best_params` of the best variant;
a plot of validation AUC against `eta0` with and without scaling.

1. `tune_base` = `Pipeline([("scale", StandardScaler()), ("features", "passthrough"), ("model", SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.01, alpha=1e-4, random_state=SEED))])`.
   It behaves like `logistic_pipe`; the `features` step only makes room for tuning.
2. **Before computing**, write down your guess: which change will help most and which will hurt most?
3. Prepare a list `manual_variants` of at least six parameter dictionaries from the table above. Each dictionary changes **one** thing.
4. For every variant: `clone(tune_base)` → `set_params(**variant)` → `fit` on training → validation AUC from `decision_function`.
   Store the results in `manual_results` sorted from best, and the best dictionary in `manual_best_params`.
5. Try one **combination** of the two best changes and add it to the table.
6. Plot: for `eta0` in 0.0001, 0.001, 0.01, 0.1, 1 compute the validation AUC with `StandardScaler()` and with `"passthrough"`
   and draw both lines in one plot with a logarithmic x axis.

**Use:** `Pipeline`, `clone`, `.set_params(**dictionary)`, `.decision_function`, `roc_auc_score`, a `for` loop,
`pd.DataFrame(list_of_dicts)`, `sort_values`, `plt.semilogx`.

<details><summary>Hint</summary>

The score for the AUC is `model.decision_function(X_val)`; it exists for both losses (the perceptron has no probabilities).
Writing `set_params(**{"model__eta0": 0.1})` is the same as `set_params(model__eta0=0.1)`; an empty dictionary `{}` keeps the base settings.
Create objects such as `PolynomialFeatures(...)` anew in every dictionary. A helper function `validate(params)` that returns the AUC
of one variant also shortens the code for the plot.

</details>

**Think about:** Which change helped and which hurt? Does it match your guess and the plots from the educational part (sections 4, 5 and 10)?
How large a difference in AUC on 40 validation defects would you consider real rather than chance?

### Example: several variants on a fixed validation set

In [ ]:
demo_base = Pipeline([("scale", StandardScaler()), ("features", "passthrough"),  # The same three steps as tune_base …
                      ("model", SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.05, random_state=7))])  # … with a constant learning rate.
demo_variants = [{}, {"model__eta0": 0.5}, {"model__alpha": 0.05}]  # An empty dictionary = base settings.
demo_rows = []  # Results of the variants.
for demo_params in demo_variants:  # One variant after another.
    demo_model = clone(demo_base).set_params(**demo_params)  # A copy of the base pipeline with one change.
    demo_model.fit(demo_X_train, demo_y_train)  # The same training data for all variants.
    demo_auc = roc_auc_score(demo_y_val, demo_model.decision_function(demo_X_val))  # AUC from the score.
    demo_rows.append({"variant": str(demo_params), "validation AUC": demo_auc})  # One table row.
display(pd.DataFrame(demo_rows).sort_values("validation AUC", ascending=False))  # Best variant on top.

**Your code:**

### 6.2 Grid search with cross-validation

**Goal / output:** `search` – a fitted `GridSearchCV` over `tune_base`; a table `grid_results` with the ten best combinations,
their mean AUC and standard deviation; a plot of the best mean AUC for each loss against `eta0`;
`manual_cv` – the mean AUC of your manual choice from the same cross-validation.

1. A dictionary `param_grid` with the five parameters from the table in step 6: 2 · 2 · 2 · 5 · 4 = 160 combinations.
2. `cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)`.
3. `search = GridSearchCV(tune_base, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)` fitted on the **development data**
   `X_dev`, `y_dev`. The test set stays aside. The computation takes a few seconds to tens of seconds.
4. Build `grid_results` from `search.cv_results_`: columns `params`, `mean_test_score`, `std_test_score`, the ten best rows.
   Print `search.best_params_` and `search.best_score_`.
5. Plot: for each loss, the best mean AUC (over the other parameters) against `eta0`.
6. Evaluate `manual_best_params` from 6.1 with the same cross-validation (`cross_val_score` on `X_dev`, `y_dev`) and store the mean in `manual_cv`.
7. Evaluate `search.best_params_` with and without `model__class_weight="balanced"` by the same cross-validation. Does the AUC change? Why so little?

**Use:** `GridSearchCV`, `StratifiedKFold`, `cross_val_score`, `search.best_params_`, `search.best_score_`,
`pd.DataFrame(search.cv_results_)`, `sort_values`, `groupby`.

<details><summary>Hint</summary>

The keys of `param_grid` are the same names as in `set_params`; the values are lists. `cv_results_` contains a column
`param_<name>` for every parameter, e.g. `param_model__eta0`. Values that are objects (scalers, feature steps) are converted to text with
`.astype(str)` before grouping. The plot can be built as `table.groupby(["eta0", "loss"])["score"].max().unstack().plot(marker="o", logx=True)`.

</details>

**Think about:** Does the best combination match your manual choice? Is the difference among the top five combinations larger than their
standard deviation? Which parameter matters most, and which hardly at all? Does it match the educational part?

### Example: a small grid on illustrative data

In [ ]:
demo_grid = {"model__alpha": [0.0001, 0.05], "model__eta0": [0.005, 0.05]}  # 2 · 2 = 4 combinations.
demo_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)  # Three folds for the small data.
demo_search = GridSearchCV(demo_base, demo_grid, scoring="roc_auc", cv=demo_cv).fit(demo_X_dev, demo_y_dev)  # Every combination on every fold.
demo_table = pd.DataFrame(demo_search.cv_results_)[["params", "mean_test_score", "std_test_score"]]  # Mean and spread over the folds.
display(demo_table.sort_values("mean_test_score", ascending=False))  # Best combination on top.
print(demo_search.best_params_, round(demo_search.best_score_, 3))  # The winner and its mean AUC.
print("mean AUC of one setting:", cross_val_score(demo_base, demo_X_dev, demo_y_dev, scoring="roc_auc", cv=demo_cv).mean().round(3))  # cross_val_score for a single setting.

**Your code:**

### 6.3 Decision threshold by the cost of errors

**Goal / output:** `chosen_params` – the parameters of the final model; `dev_proba` – out-of-fold probabilities of a defect for all
development parts; a table `threshold_costs` with the total cost for thresholds 0.01, 0.02, …, 0.99; the cheapest threshold
`chosen_threshold`; a plot of the cost (and of precision and recall) against the threshold.

1. The threshold needs **probabilities**, so the final model must be a logistic neuron. `chosen_params` = `search.best_params_`
   if its `model__loss` is `"log_loss"`; otherwise take the best `"log_loss"` row of all combinations, `pd.DataFrame(search.cv_results_)`.
2. `dev_proba = cross_val_predict(clone(tune_base).set_params(**chosen_params), X_dev, y_dev, cv=cv, method="predict_proba")[:, 1]`.
   Every part gets a probability from a model that did not see it during learning.
3. For every threshold t compute the predictions `dev_proba >= t` and their total cost with `evaluate`. Store the table in `threshold_costs`
   and the cheapest threshold in `chosen_threshold`.
4. Compare `chosen_threshold` with the theoretical value C_FP / (C_FP + C_FN) = 1/11 ≈ 0.091 and the cost at 0.5.
5. Plot the cost against the threshold; in a second plot draw precision and recall against the threshold.
6. Repeat steps 2 and 3 with `model__class_weight="balanced"` added to a copy of `chosen_params` (keep `chosen_params` itself unchanged).
   Where is the cheapest threshold now, and why? Is the minimum cost lower?

**Use:** `cross_val_predict`, a `for` loop over `np.round(np.arange(0.01, 1.0, 0.01), 2)`, `evaluate`, `pd.DataFrame`, `idxmin`, `plt.plot`.

<details><summary>Hint</summary>

The best combination with a given loss: `table = pd.DataFrame(search.cv_results_)`, then `table[table["param_model__loss"] == "log_loss"].sort_values("mean_test_score").iloc[-1]["params"]`.
Out-of-fold probabilities come from five models, each fitted on four fifths of `X_dev`; their threshold is therefore chosen without the test set
and without the optimism of training predictions. Collect rows `{"threshold": t, **evaluate(y_dev, dev_proba >= t)}` and build the table from them.

</details>

**Think about:** How far is the cheapest threshold from 0.5? How many defects do you miss and how many good parts do you send to the check at
this threshold? Why does the theoretical value 1/11 hold only approximately here?

### Example: cost for a few thresholds

In [ ]:
demo_dev_proba = cross_val_predict(demo_base, demo_X_dev, demo_y_dev, cv=demo_cv, method="predict_proba")[:, 1]  # Probabilities from models that did not see the sample.
demo_rows = [{"threshold": t, **evaluate(demo_y_dev, demo_dev_proba >= t)} for t in (0.05, 0.1, 0.3, 0.5)]  # One row per threshold.
demo_costs = pd.DataFrame(demo_rows).set_index("threshold")  # Thresholds as the row index.
display(demo_costs.round(3))  # The table.
print("cheapest of these thresholds:", demo_costs["cost"].idxmin())  # Row with the lowest cost.

**Your code:**

## 7. Freezing the choice and the final test

**Goal / output:** `final_pipeline` – `tune_base` with `chosen_params`, fitted on the whole development data;
the test probabilities `test_proba`; the test decisions `test_pred` at `chosen_threshold`; a table `final_table` comparing
the model at `chosen_threshold`, the same model at 0.5 and “always good” on the test set; a confusion matrix.

The choice is frozen: parameters from 6.2, threshold from 6.3. Create `final_pipeline` as `clone(tune_base)` with `chosen_params`,
fit it on `X_dev`, `y_dev` (1500 parts) and only then predict `X_test` once.

**Use:** `clone`, `.set_params(**...)`, `.fit`, `.predict_proba`, `evaluate`, `ConfusionMatrixDisplay.from_predictions`.

<details><summary>Hint</summary>

For “always good” use `np.zeros(len(y_test), dtype=int)`. Test targets are used for evaluation only.

</details>

**Once the test results are displayed, only interpretation follows.** Changing the threshold or the parameters based on test results
would turn the test set into another validation set.

**Think about:** How many defects did the model miss on the test set and how many good parts did it send to the check?
How much does the chosen threshold save compared with 0.5 and with no model? Is the test AUC close to `search.best_score_`?

### Example: refitting the frozen choice and one test

In [ ]:
demo_final = clone(demo_base).set_params(**demo_search.best_params_).fit(demo_X_dev, demo_y_dev)  # Frozen settings, all development data.
demo_test_proba = demo_final.predict_proba(demo_X_test)[:, 1]  # The only prediction of the test set.
display(pd.DataFrame([evaluate(demo_y_test, demo_test_proba >= 0.1)], index=["threshold 0.1"]).round(3))  # Evaluated once.

**Your code:**